In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install joblib reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 42.2 MB/s eta 0:00:00


In [3]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

In [4]:
PROJECT_PATH = "/content/drive/MyDrive/VFD_Testbench_Project/"

In [5]:
anomaly_model = joblib.load(PROJECT_PATH + "anomaly_model.pkl")
health_model  = joblib.load(PROJECT_PATH + "health_model.pkl")
fault_model   = joblib.load(PROJECT_PATH + "fault_model.pkl")
rul_model     = joblib.load(PROJECT_PATH + "rul_model_100k.pkl")

anomaly_scaler = joblib.load(PROJECT_PATH + "anomaly_scaler.pkl")
health_scaler  = joblib.load(PROJECT_PATH + "health_scaler.pkl")
fault_scaler   = joblib.load(PROJECT_PATH + "fault_scaler.pkl")

print("All models loaded successfully.")

All models loaded successfully.


In [6]:
anomaly_features = [
 'ia','ib','idc','vdc',
 'current_imbalance',
 'idc_rateofchange','vdc_rateofchange',
 'power_ac','t1','t2','t3','temp_diff_max'
]

health_features = anomaly_features
fault_features = anomaly_features

rul_features = [
 'anomaly_score',
 'orca_temp_inv',
 'orca_i_mag',
 'orca_v_dc',
 'orca_p_out'
]

In [7]:
def extract_features(window_df, feature_list):

    features = []

    for col in feature_list:
        values = window_df[col].values

        features.append(np.mean(values))
        features.append(np.std(values))
        features.append(np.max(values))
        features.append(np.min(values))
        features.append(values[-1] - values[0])  # slope

    return np.array(features).reshape(1, -1)

In [8]:
def compute_rul(window_df):

    features = []

    for col in rul_features:
        values = window_df[col].values

        features.append(np.mean(values))
        features.append(np.std(values))
        features.append(np.max(values))
        features.append(np.min(values))
        features.append(np.median(values))
        features.append(values[-1] - values[0])

    X_rul = np.array(features).reshape(1, -1)

    return rul_model.predict(X_rul)[0]

In [20]:
# STEP 10 — Create Demo Dataset for Testing

# Load master fault dataset
df_master = pd.read_csv(PROJECT_PATH + "orca_vfd_fault_master.csv")

print("Original dataset shape:", df_master.shape)

# Take last 1200 rows to simulate live data stream
df_demo = df_master.tail(1200).copy()

# Remove training-only columns if present
df_demo = df_demo.drop(columns=["fault_code"], errors="ignore")

# Save as demo file
df_demo.to_csv(PROJECT_PATH + "demo_live_input.csv", index=False)

print("Demo dataset created successfully.")
print("Demo dataset shape:", df_demo.shape)

Original dataset shape: (10892, 29)
Demo dataset created successfully.
Demo dataset shape: (1200, 29)


In [21]:
# STEP 10B — Create Lifecycle Demo Dataset for RUL

df_lifecycle = pd.read_csv(PROJECT_PATH + "orca_vfd_full_lifecycle_100k.csv")

print("Lifecycle dataset shape:", df_lifecycle.shape)

# Ensure chronological order
df_lifecycle = df_lifecycle.sort_values("life_time_hours")

# Take last 2000 rows (near failure region)
df_lifecycle_demo = df_lifecycle.tail(2000).copy()

# Remove columns not needed for deployment
df_lifecycle_demo = df_lifecycle_demo.drop(
    columns=["life_time_hours", "lifecycle_phase", "orca_fault_code"],
    errors="ignore"
)

# Save demo file
df_lifecycle_demo.to_csv(PROJECT_PATH + "demo_lifecycle_input.csv", index=False)

print("Lifecycle demo dataset created successfully.")
print("Lifecycle demo shape:", df_lifecycle_demo.shape)

Lifecycle dataset shape: (32751, 11)
Lifecycle demo dataset created successfully.
Lifecycle demo shape: (2000, 8)


In [22]:
# STEP 11 — Load Lifecycle Demo Data

df_live = pd.read_csv(PROJECT_PATH + "demo_lifecycle_input.csv")

print("Loaded lifecycle demo shape:", df_live.shape)

Loaded lifecycle demo shape: (2000, 8)


In [23]:
# STEP 12 — Load Fault Demo Dataset

df_fault_live = pd.read_csv(PROJECT_PATH + "demo_live_input.csv")

print("Fault demo dataset shape:", df_fault_live.shape)
print("Columns available:")
print(df_fault_live.columns)

Fault demo dataset shape: (1200, 29)
Columns available:
Index(['ia', 'ib', 'vdc', 'idc', 't1', 't2', 't3', 'vd', 'fdd', 'timestamp',
       'ia_original', 'ia_arduino', 'ib_original', 'ib_arduino',
       'idc_original', 'idc_arduino', 'power_dc', 'power_ac',
       'current_imbalance', 'temp_diff_max', 'ia_normalized', 'ib_normalized',
       'vdc_rateofchange', 'idc_rateofchange', 'vdc_movingavg',
       'idc_movingavg', 'orca_fault_code', 'orca_is_normal',
       'orca_fault_group'],
      dtype='object')


In [56]:
def run_diagnostics(df_fault, df_lifecycle):

    report = {}

    row = df_fault.iloc[-1]

    # ---------------- MODULE A ----------------
    anomaly_features = anomaly_scaler.feature_names_in_
    X_anom_raw = pd.DataFrame([row[anomaly_features].values],
                              columns=anomaly_features)
    X_anom = anomaly_scaler.transform(X_anom_raw)
    anomaly_output = anomaly_model.predict(X_anom)[0]

    if anomaly_output == 1:
        report["System Status"] = "Normal"
        report["Health"] = "Healthy"
        report["Fault"] = "None"
        report["RUL (hours)"] = "Not Required"
        report["Drive Score (%)"] = 100
        return report

    report["System Status"] = "Anomaly Detected"

    # ---------------- MODULE C ----------------
    health_features = health_scaler.feature_names_in_
    X_health_raw = pd.DataFrame([row[health_features].values],
                                columns=health_features)
    X_health = health_scaler.transform(X_health_raw)
    health_output = health_model.predict(X_health)[0]

    if health_output:
        report["Health"] = "Minor Degradation"
        report["Fault"] = "None"
        report["RUL (hours)"] = "Monitoring"
        report["Drive Score (%)"] = 75
        return report

    report["Health"] = "Critical"

    # ---------------- MODULE D ----------------
    fault_features = fault_scaler.feature_names_in_
    X_fault_raw = pd.DataFrame([row[fault_features].values],
                               columns=fault_features)
    X_fault = fault_scaler.transform(X_fault_raw)
    fault_code = fault_model.predict(X_fault)[0]

    report["Fault Code"] = fault_code

    fault_mapping = {
        0: "Bearing Fault",
        1: "DC Bus Ripple",
        2: "Overload",
        3: "Misalignment",
        4: "IGBT Failure",
        5: "Cooling Failure",
        6: "Gate Drive Issue",
        7: "Short Circuit",
        8: "Unknown Fault"
    }

    report["Fault"] = fault_mapping.get(fault_code, "Unknown Fault")

    # ---------------- MODULE B (RUL) ----------------
    rul_value = compute_rul(df_lifecycle.tail(1000))

    report["RUL (hours)"] = round(rul_value, 2)
    CRITICAL_THRESHOLD = 20000

    drive_score = max(0, min(100, (rul_value / CRITICAL_THRESHOLD) * 100))


    report["Drive Score (%)"] = round(drive_score, 2)

    return report

In [45]:
report_final = run_diagnostics(df_fault_live, df_live)

In [46]:
def generate_full_feature_dict(window_df):

    feature_dict = {}

    base_columns = [
        'ia','ib','idc','vdc',
        'power_ac','power_dc',
        't1','t2','t3',
        'current_imbalance'
    ]

    for col in base_columns:

        if col in window_df.columns:

            values = window_df[col].values

            feature_dict[f"{col}_mean"] = np.mean(values)
            feature_dict[f"{col}_std"] = np.std(values)
            feature_dict[f"{col}_max"] = np.max(values)
            feature_dict[f"{col}_min"] = np.min(values)
            feature_dict[f"{col}_slope"] = values[-1] - values[0]

    return feature_dict

In [47]:
print("Anomaly expects:", anomaly_scaler.n_features_in_)
print("Health expects:", health_scaler.n_features_in_)
print("Fault expects:", fault_scaler.n_features_in_)

Anomaly expects: 25
Health expects: 24
Fault expects: 24


In [48]:
print("Anomaly feature names:")
print(anomaly_scaler.feature_names_in_)

print("\nHealth feature names:")
print(health_scaler.feature_names_in_)

print("\nFault feature names:")
print(fault_scaler.feature_names_in_)

Anomaly feature names:
['ia' 'ib' 'vdc' 'idc' 't1' 't2' 't3' 'vd' 'timestamp' 'ia_original'
 'ia_arduino' 'ib_original' 'ib_arduino' 'idc_original' 'idc_arduino'
 'power_dc' 'power_ac' 'current_imbalance' 'temp_diff_max' 'ia_normalized'
 'ib_normalized' 'vdc_rateofchange' 'idc_rateofchange' 'vdc_movingavg'
 'idc_movingavg']

Health feature names:
['ia' 'ib' 'vdc' 'idc' 't1' 't2' 't3' 'vd' 'ia_original' 'ia_arduino'
 'ib_original' 'ib_arduino' 'idc_original' 'idc_arduino' 'power_dc'
 'power_ac' 'current_imbalance' 'temp_diff_max' 'ia_normalized'
 'ib_normalized' 'vdc_rateofchange' 'idc_rateofchange' 'vdc_movingavg'
 'idc_movingavg']

Fault feature names:
['ia' 'ib' 'vdc' 'idc' 't1' 't2' 't3' 'vd' 'ia_original' 'ia_arduino'
 'ib_original' 'ib_arduino' 'idc_original' 'idc_arduino' 'power_dc'
 'power_ac' 'current_imbalance' 'temp_diff_max' 'ia_normalized'
 'ib_normalized' 'vdc_rateofchange' 'idc_rateofchange' 'vdc_movingavg'
 'idc_movingavg']


In [49]:
print(run_diagnostics)

<function run_diagnostics at 0x7ca3a6c5b380>


In [57]:
report_final = run_diagnostics(df_fault_live, df_live)

print("\n===== FINAL INTEGRATED REPORT =====\n")
for k, v in report_final.items():
    print(f"{k}: {v}")


===== FINAL INTEGRATED REPORT =====

System Status: Anomaly Detected
Health: Critical
Fault Code: 8
Fault: Unknown Fault
RUL (hours): 957.62
Drive Score (%): 4.79


In [63]:
pip install reportlab

In [68]:
import matplotlib.pyplot as plt
import numpy as np

def generate_rul_trend(df_lifecycle, window_size=1000, step=200):

    rul_values = []
    indices = []

    for i in range(window_size, len(df_lifecycle), step):
        window = df_lifecycle.iloc[i-window_size:i]
        rul_pred = compute_rul(window)
        rul_values.append(rul_pred)
        indices.append(i)

    plt.figure(figsize=(8,4))
    plt.plot(indices, rul_values)
    plt.title("RUL Trend Over Time")
    plt.xlabel("Time Index")
    plt.ylabel("Predicted RUL (hours)")
    plt.grid(True)

    trend_path = PROJECT_PATH + "rul_trend.png"
    plt.savefig(trend_path)
    plt.close()

    return trend_path

In [69]:
import matplotlib.pyplot as plt
import numpy as np

def generate_rul_trend(df_lifecycle, window_size=1000, step=200):

    rul_values = []
    indices = []

    for i in range(window_size, len(df_lifecycle), step):
        window = df_lifecycle.iloc[i-window_size:i]
        rul_pred = compute_rul(window)
        rul_values.append(rul_pred)
        indices.append(i)

    plt.figure(figsize=(8,4))
    plt.plot(indices, rul_values)
    plt.title("RUL Trend Over Time")
    plt.xlabel("Time Index")
    plt.ylabel("Predicted RUL (hours)")
    plt.grid(True)

    trend_path = PROJECT_PATH + "rul_trend.png"
    plt.savefig(trend_path)
    plt.close()

    return trend_path

In [70]:
rul_trend_image = generate_rul_trend(df_live)

In [71]:
model_performance = {
    "Anomaly Model": "IsolationForest (Unsupervised)",
    "Health Model Accuracy": "95%",
    "Fault Model Accuracy": "93%",
    "RUL RMSE": "2501 hours",
    "RUL R2 Score": "0.99"
}

In [72]:
def generate_recommendation(report):

    rul = report.get("RUL (hours)", None)

    if isinstance(rul, str):
        return "No maintenance required."

    if rul < 2000:
        return "Immediate maintenance required. High failure risk."
    elif rul < 10000:
        return "Schedule maintenance within next operating cycle."
    else:
        return "System operating within safe limits."

In [73]:
from reportlab.platypus import Image

def generate_pdf_report(report, save_path, rul_image_path, performance_dict):

    doc = SimpleDocTemplate(save_path)
    elements = []
    styles = getSampleStyleSheet()

    elements.append(Paragraph("<b>VFD DRIVE DIAGNOSTIC REPORT</b>", styles["Title"]))
    elements.append(Spacer(1, 20))

    # Diagnostic Table
    data = [["Parameter", "Value"]]
    for key, value in report.items():
        data.append([key, str(value)])

    table = Table(data, colWidths=[200, 250])
    table.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.grey),
        ("TEXTCOLOR", (0,0), (-1,0), colors.whitesmoke),
        ("GRID", (0,0), (-1,-1), 1, colors.black),
        ("BACKGROUND", (0,1), (-1,-1), colors.beige),
    ]))

    elements.append(table)
    elements.append(Spacer(1, 30))

    # Add RUL Trend Graph
    elements.append(Paragraph("<b>RUL Trend Analysis</b>", styles["Heading2"]))
    elements.append(Spacer(1, 10))
    elements.append(Image(rul_image_path, width=400, height=200))
    elements.append(Spacer(1, 30))

    # Add Performance Metrics
    elements.append(Paragraph("<b>Model Performance Summary</b>", styles["Heading2"]))
    elements.append(Spacer(1, 10))

    perf_data = [["Metric", "Value"]]
    for key, value in performance_dict.items():
        perf_data.append([key, value])

    perf_table = Table(perf_data, colWidths=[200, 250])
    perf_table.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.darkblue),
        ("TEXTCOLOR", (0,0), (-1,0), colors.whitesmoke),
        ("GRID", (0,0), (-1,-1), 1, colors.black),
    ]))

    elements.append(perf_table)

    doc.build(elements)

    print("Enhanced PDF Report Generated Successfully!")

In [74]:
pdf_path = PROJECT_PATH + "Drive_Diagnostic_Report_Enhanced.pdf"

generate_pdf_report(
    report_final,
    pdf_path,
    rul_trend_image,
    model_performance
)

Enhanced PDF Report Generated Successfully!
